# Mini-project — Gas liquefaction (LNG) 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import caldis
from CoolProp.CoolProp import PropsSI

from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.components import ControlVolume
from caldis.components import SizingHX
from caldis.components import FlashTank
from caldis.components import Valve
from caldis.components import Splitter
from caldis.components import Mixer
from caldis.components import Turbine
from caldis.components.combustion.heater import SimpleHeater   # used here as a cooler
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve                 # one-call workflow (DOF -> start -> solve)
from caldis.solvers.autostart import autostart, fix_port
from caldis.solvers.dynamic import simulate

In [ ]:
# Constants
gas = "Methane"
methane_be = CoolPropBackend(gas)
T0 = 273.15 + 25.0  # K
P0 = 1e5 # Pa
P_high = 100e5
ic_T_out = T0 + 5 # K
comp_is_e = 0.75 
HX_pinch = 5 # K
HP_recup_coeff = 1000 # W/m2K
LP_recup_coeff = 500 # W/m2K
mdot = 1.0 # kg/s
comp_r = 3.16 
x_e = 0.5 # fraction of flow to expander

In [11]:
# Components

## Pinch ??? HX pinch ? 
## rendement turbine ? 

meth_inlet = FluidBoundary(name="Methane_inlet", backend=methane_be, T=T0, p=P0, mdot=-mdot)
mix_in = Mixer(name="Mixer_in", backend=methane_be)
comp_1 = Compressor(name="Compressor_1", backend=methane_be, eta_is=comp_is_e)
comp_2 = Compressor(name="Compressor_2", backend=methane_be, eta_is=comp_is_e)
comp_3 = Compressor(name="Compressor_3", backend=methane_be, eta_is=comp_is_e)
comp_4 = Compressor(name="Compressor_4", backend=methane_be, eta_is=comp_is_e)
ic_1 = SimpleHeater(name="Intercooler_1", backend=methane_be)
ic_2 = SimpleHeater(name="Intercooler_2", backend=methane_be)
ic_3 = SimpleHeater(name="Intercooler_3", backend=methane_be)
ac = SimpleHeater(name="Aftercooler", backend=methane_be)
splitter = Splitter(name="Splitter", backend=methane_be)
expander = Turbine(name="Expander", backend=methane_be, eta_is=0.8)
recup = SizingHX(name="Recuperator", backend_hot=methane_be, backend_cold=methane_be, h_hot=HP_recup_coeff, h_cold=LP_recup_coeff, area_ratio_cold_to_hot=1.0)
valve = Valve(name="Valve", backend=methane_be)
mix_out = Mixer(name="Mixer_out", backend=methane_be)
flash = FlashTank(name="Flash", backend=methane_be)
meth_outlet = FluidBoundary(name="Methane_outlet", backend=methane_be)



In [ ]:
#System definition 
sys = System(name="LNG_L0")

sys.add(mix_in, comp_1, ic_1, comp_2, ic_2, comp_3, ic_3, comp_4, ac, splitter, expander, recup, valve, mix_out, flash)
sys.connect(meth_inlet.port, mix_in.in1)
sys.connect(mix_in.out, comp_1.inl)
sys.connect(comp_1.out, ic_1.inl)
sys.connect(ic_1.out, comp_2.inl)
sys.connect(comp_2.out, ic_2.inl)
sys.connect(ic_2.out, comp_3.inl)
sys.connect(comp_3.out, ic_3.inl)
sys.connect(ic_3.out, comp_4.inl)
sys.connect(comp_4.out, ac.inl)
sys.connect(ac.out, splitter.inl)

# splitter's 1-x_e
sys.connect(splitter.out1, recup.cold_in)
sys.connect(recup.cold_out, valve.inl)
sys.connect(valve.out, mix_out.in1)
##
# splitter's x_e
sys.connect(splitter.out2, expander.inl)
sys.connect(expander.out, mix_out.in2)
##

sys.connect(mix_out.out, flash.inl)

sys.connect(flash.liq, meth_outlet.port)
sys.connect(flash.vap, recup.hot_in)
sys.connect(recup.hot_out, mix_in.in2)

sys.add_constraint(lambda : comp_1.out.p - comp_r*P0, name="comp_1_pressure_constraint")
sys.add_constraint(lambda : comp_2.out.p - comp_r*comp_1.out.p, name="comp_2_pressure_constraint")
sys.add_constraint(lambda : comp_3.out.p - comp_r*comp_2.out.p, name="comp_3_pressure_constraint")
sys.add_constraint(lambda : comp_4.out.p - comp_r*comp_3.out.p, name="comp_4_pressure_constraint")
sys.add_constraint(lambda : ic_1.out.T - ic_T_out, name="ic_1_temperature_constraint")
sys.add_constraint(lambda : ic_2.out.T - ic_T_out, name="ic_2_temperature_constraint")
sys.add_constraint(lambda : ic_3.out.T - ic_T_out, name="ic_3_temperature_constraint")
sys.add_constraint(lambda : ac.out.T - ic_T_out, name="ac_temperature_constraint")
#ajouter flash à 1 bar 
#ajouter le pich 


<function __main__.<lambda>()>

In [13]:
sol = solve(sys)

✗ DOF: [DOF sur-spécifié] 115 inconnues, 116 équations : 1 équation(s) de trop (condition imposée en double ?).
  -> the model is not square. Add/remove a constraint or boundary spec (MODELLING issue, not a start issue).
